In [1]:

import pandas as pd
from pathlib import Path

# Load the original dataset again
DATA_PATH = Path("../data/spam.csv")
df = pd.read_csv(DATA_PATH, encoding="latin-1")

# Check the dataset
print("Original dataset shape:", df.shape)
df.head()


Original dataset shape: (5572, 5)


,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [2]:

# Count rows that are exact duplicates across all columns
print("Exact duplicate rows:", df.duplicated().sum())

# Count repeated SMS message texts
print("Duplicate message texts:", df["v2"].duplicated().sum())

# Check whether the same message text has conflicting labels
label_counts_per_message = df.groupby("v2")["v1"].nunique()

conflicting_messages = (label_counts_per_message > 1).sum()

print("Messages with conflicting labels:", conflicting_messages)


Exact duplicate rows: 403
Duplicate message texts: 403
Messages with conflicting labels: 0


In [3]:

# Remove exact duplicate rows
df = df.drop_duplicates().reset_index(drop=True)

# Check the dataset after removing duplicates
print("Shape after removing duplicates:", df.shape)
print("Remaining duplicate rows:", df.duplicated().sum())


Shape after removing duplicates: (5169, 5)
Remaining duplicate rows: 0


In [4]:

# Keep only the label and SMS message columns
df = df[["v1", "v2"]].copy()

# Rename columns to make their purpose clearer
df = df.rename(columns={
    "v1": "label",
    "v2": "message"
})

# Inspect the cleaned dataset
print("Dataset shape:", df.shape)
print("\nColumn names:", df.columns.tolist())

df.head()


Dataset shape: (5169, 2)

Column names: ['label', 'message']


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [5]:

# Check the unique target labels
print("Unique labels:", df["label"].unique())

# Count messages in each category
print("\nLabel counts:")
print(df["label"].value_counts())

# Calculate the percentage of each category
print("\nLabel percentages:")
print((df["label"].value_counts(normalize=True) * 100).round(2))


Unique labels: <StringArray>
['ham', 'spam']
Length: 2, dtype: str

Label counts:
label
ham     4516
spam     653
Name: count, dtype: int64

Label percentages:
label
ham     87.37
spam    12.63
Name: proportion, dtype: float64


In [6]:

# Convert text labels into numerical labels
df["label"] = df["label"].map({
    "ham": 0,
    "spam": 1
})

# Verify the encoding
print("Unique encoded labels:", df["label"].unique())
print("\nLabel counts after encoding:")
print(df["label"].value_counts())

df.head()


Unique encoded labels: [0 1]

Label counts after encoding:
label
0    4516
1     653
Name: count, dtype: int64


,label,message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [7]:

# Check for missing values in each column
print("Missing values:")
print(df.isnull().sum())

# Check for empty or whitespace-only messages
empty_messages = df["message"].str.strip().eq("").sum()

print("\nEmpty messages:", empty_messages)

# Check whether any labels are missing
print("Missing labels:", df["label"].isnull().sum())


Missing values:
label      0
message    0
dtype: int64

Empty messages: 0
Missing labels: 0


In [8]:

from sklearn.model_selection import train_test_split

# Separate input messages (X) from target labels (y)
X = df["message"]
y = df["label"]

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Check the resulting dataset sizes
print("Training messages:", len(X_train))
print("Testing messages:", len(X_test))

# Check class distribution in each set
print("\nTraining label distribution:")
print(y_train.value_counts(normalize=True).mul(100).round(2))

print("\nTesting label distribution:")
print(y_test.value_counts(normalize=True).mul(100).round(2))


Training messages: 4135
Testing messages: 1034

Training label distribution:
label
0    87.38
1    12.62
Name: proportion, dtype: float64

Testing label distribution:
label
0    87.33
1    12.67
Name: proportion, dtype: float64


In [9]:

from pathlib import Path

# Create a folder for processed data
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Combine messages and labels for each split
train_df = pd.DataFrame({
    "message": X_train,
    "label": y_train
})

test_df = pd.DataFrame({
    "message": X_test,
    "label": y_test
})

# Save the datasets as CSV files
train_df.to_csv(PROCESSED_DIR / "train.csv", index=False)
test_df.to_csv(PROCESSED_DIR / "test.csv", index=False)

print("Training data saved:", len(train_df), "rows")
print("Testing data saved:", len(test_df), "rows")
print("Saved in:", PROCESSED_DIR)


Training data saved: 4135 rows
Testing data saved: 1034 rows
Saved in: ../data/processed


### Data Preprocessing — Summary

1. Dataset Cleaning

>>Loaded the original SMS Spam Collection dataset containing 5,572 rows.

>>Identified and removed 403 duplicate records.

>>Retained the relevant label and message columns.

>>Renamed the columns for clarity.

>>Checked for missing values and empty messages; none were found.

## 2. Label Encoding

Converted the categorical labels into numerical values:

ham → 0

spam → 1



## 3. Train-Test Split

>>Split the cleaned dataset into training and testing sets using an 80:20 ratio.

>>Training set: 4,135 messages

>>Testing set: 1,034 messages

>>Used random_state=42 for reproducibility.

>>Used stratified sampling to preserve the class distribution.

### 4. Class Imbalance

The cleaned dataset contains 4,516 ham messages (87.37%) and 653 spam messages (12.63%). We retained the original distribution and will establish a baseline before deciding whether additional imbalance-handling techniques are necessary.

### 5. Saved Datasets

Saved the processed datasets to:

data/processed/train.csv

data/processed/test.csv

### 6.Conclusion





The dataset is cleaned, validated, encoded, and split. The next stage is feature engineering, where we will transform SMS text into numerical features using TF-IDF. The vectorizer will be fitted only on the training messages to avoid data leakage.